# DecoderTCR Quick Start

This notebook demonstrates how to use DecoderTCR for:
1. TCR-pMHC interaction prediction
2. pMHC binding prediction

## Setup

Set the environment variable for model caching (optional):

In [ ]:
import os
os.environ['TORCH_HUB_DIR'] = '<cache path>'  # Set your cache directory

## 1. TCR-pMHC Interaction Prediction

Predict TCR-pMHC binding using interaction scores (comparing TCR+pMHC vs pMHC alone).

In [ ]:
from DecoderTCR.utils.predict_TpM import load_model, predict_single

# Load model from checkpoint
checkpoint_path = '<path to checkpoint>'  # Set your checkpoint path
model = load_model(checkpoint_path=checkpoint_path, device='cuda:0')

In [ ]:
import torch

# Example sample
sample = {
    'HLA_seq': 'GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWEPSSQPTIPIVGIIAGLVLFGAVITGAVVAAVMWRRKSSDRKGGSYSQAASSDSAQGSDVSLTACKVMIQRTPKIQVYSRHPAENGKSNFLNCYVSGFHPSDIEVDLLKNGERIEKVEHSDLSFSKDWSFYLLYYTEFTPTEKDEYACRVNHVTLSQPKIVKWDRDM',
    'epitope': 'YLQPRTFLL',
    'TCR_seq': 'MISLRVLLVILWLQLSWVWSQRKEVEQDPGPFNVPEGATVAFNCTYSNSASQSFFWYRQDCRKEPKLLMSVYSSGNEDGRFTAQLNRASQYISLLIRDSKLSDSATYLCVVNIDTDKLIFGTGTRLQVFPNIQNPDPAVYQLRDSKSSDKSVCLFTDFDSQTNVSQSKDSDVYITDKTVLDMRSMDFKSNSAVAWSNKSDFACANAFNNSIIPEDTFFPSPESSCDVKLVEKSFETDTNLNFQNLSVIGFRILLLKVAGFNLLMTLRLWSSMISLRVLLVILWLQLSWVWSQRKEVEQDPGPFNVPEGATVAFNCTYSNSASQSFFWYRQDCRKEPKLLMSVYSSGNEDGRFTAQLNRASQYISLLIRDSKLSDSATYLCVVNIDTDKLIFGTGTRLQVFPNIQNPDPAVYQLRDSKSSDKSVCLFTDFDSQTNVSQSKDSDVYITDKTVLDMRSMDFKSNSAVAWSNKSDFACANAFNNSIIPEDTFFPSPESSCDVKLVEKSFETDTNLNFQNLSVIGFRILLLKVAGFNLLMTLRLWSSMDTWLVCWAIFSLLKAGLTEPEVTQTPSHQVTQMGQEVILRCVPISNHLYFYWYRQILGQKVEFLVSFYNNEISEKSEIFDDQFSVERPDGSNFTLKIRSTKLEDSAMYFCATGGDHNTGELFFGEGSRLTVLEDLNKVFPPEVAVFEPSEAEISHTQKATLVCLATGFFPDHVELSWWVNGKEVHSGVSTDPQPLKEQPALNDSRYCLSSRLRVSATFWQNPRNHFRCQVQFYGLSENDEWTQDRAKPVTQIVSAEAWGRADCGFTSVSYQQGVLSATILYEILLGKATLYAVLVSALVLMAMVKRKDF'
}

# Predict
with torch.no_grad():
    score = predict_single(model, sample, device='cuda:0')

print(f"TCR-pMHC Interaction Score: {score:.4f}")

## 2. pMHC Binding Prediction

Score epitope-HLA binding using span pseudo-likelihood (no TCR required).

In [ ]:
from DecoderTCR.utils.predict_pMHC import load_model as load_model_pMHC, predict_single as predict_single_pMHC

# Load model (can reuse the same checkpoint)
model_pMHC = load_model_pMHC(checkpoint_path=checkpoint_path, device='cuda:0')

In [ ]:
# Example sample (no TCR needed)
sample_pMHC = {
    'HLA_seq': 'GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE',
    'epitope': 'GILGFVFTL'
}

# Predict
with torch.no_grad():
    score_pMHC = predict_single_pMHC(model_pMHC, sample_pMHC, device='cuda:0')

print(f"pMHC Binding Score (Pseudo-likelihood): {score_pMHC:.4f}")

## 3. pMHC embeddings
attain sequence level embeddings from DecoderTCR

In [ ]:
from DecoderTCR.model.DecoderTCR import DecoderTCRModel

checkpoint_path = '<path to checkpoint>'  # Set your checkpoint path
model = DecoderTCRModel.load_from_checkpoint(checkpoint_path = checkpoint_path, base_model = 'ESM2_3B')

In [ ]:
from DecoderTCR.utils.tokenizer import tokenize


sample_pMHC = {
    'HLA_seq': 'GSHSMRYFFTSVSRPGRGEPRFIAVGYVDDTQFVRFDSDAASQRMEPRAPWIEQEGPEYWDGETRKVKAHSQTHRVDLGTLRGYYNQSEAGSHTVQRMYGCDVGSDWRFLRGYHQYAYDGKDYIALKEDLRSWTAADMAAQTTKHKWEAAHVAEQLRAYLEGTCVEWLRRYLENGKETLQRTDAPKTHMTHHAVSDHEATLRCWALSFYPAEITLTWQRDGEDQTQDTELVETRPAGDGTFQKWAAVVVPSGQEQRYTCHVQHEGLPKPLTLRWE',
    'epitope': 'GILGFVFTL'
}

input_token = tokenize(sample_pMHC['HLA_seq'] + sample_pMHC['epitope']).cuda(0)
embed = model.get_embeddings(input_token)
print(embed.size())